In [20]:
import jax 
import jax.numpy as jnp 

from functools import partial

jax.config.update("jax_enable_x64", True)

# Adjoints

We have the following situation with an objective
$$f(\theta,x)$$
and a PDE constrained $x$:

$$A(\theta)x = b(\theta)$$

or equivalently:

$$r(\theta,x) = A(\theta)x -b(\theta) = 0$$


Furthermore, we define 

$$A^T \psi = \left[\frac{\partial f}{\partial x}\right]^T$$

(the transpose is because multiple f values mean multiple solves). Then, we have 
$$\frac{df}{d\theta} = \frac{\partial f}{\partial \theta } - \psi^T \frac{\partial r}{\partial \theta}$$
This has two different components:
1. The direct dependence of f on $\theta$
2. The indirect dependence of $f$ on $\theta$ through the state equation. This is a Vector-Jacobian Product (VJP) of the adjoint with the residual Jacobian.


Note that JAX has implemented the linear solve JVP itself; so actually there is no need to do this in simple cases, however, we will later use a operator definition of $A(\theta)$, and especially $A^{-1}(\theta)$, which we *cannot* differentiate through.
This is implemented in the below example:

In [131]:

def f(x, theta):
    return jnp.sum(x ** 2) + theta[0] * jnp.sum(x) +x[0]*theta[1] # arbitrary objective

def A_param(theta):
    return jnp.array([[1.0, theta[0]],[theta[1], 1.0]])

def b_param(theta):
    return jnp.array([1.0 - 0.1**theta[1], 1.0 - 0.2**theta[0]])

def solve(A,b):
    return jnp.linalg.solve(A,b)

def solve_transpose(A,b):
    return jnp.linalg.solve(A.T,b)


def residual(x, theta):
    return A_param(theta) @ x - b_param(theta)
    
def df_dtheta(theta):
    A = A_param(theta)
    b = b_param(theta)
    x = solve(A,b)
    pf_ptheta = jax.grad(f, argnums=1)(x, theta)
    pf_px     = jax.grad(f, argnums=0)(x, theta)
    psi = solve_transpose(A, pf_px)    
    _, vj_thing = jax.vjp(partial(residual,x),theta)    
    vjp_=  vj_thing( - psi)
    return pf_ptheta + vjp_[0]

theta_base = jnp.array([0.5, 0.3])

f_t = lambda theta: f(solve(A_param(theta), b_param(theta)), theta)
print("f:                            ", f_t(theta_base))
print("df_dtheta using adjoint:      ", df_dtheta(theta_base))
print("df_dtheta using JAX gradient: ", jax.grad(f_t)(theta_base))


f:                             0.7398966031191584
df_dtheta using adjoint:       [0.906369   1.22608691]
df_dtheta using JAX gradient:  [0.906369   1.22608691]


Now, in the future, we want to be able to compose the function:

$G(\theta) = g(\theta) \circ f(\theta)$

and be able to do 
$\frac{dG}{d\theta}$ 
without writing a custom adjoint again.


[Again there is no need to do this for this simple example; JAX already has a custom VJP and JVP implemented for a jnp.linalg.solve]

In [136]:
@partial(jax.custom_vjp)
def F(theta):
    return f(solve(A_param(theta), b_param(theta)), theta)


def F_fwd(theta):
    A = A_param(theta)
    b = b_param(theta)
    x = solve(A,b)
    y = f(x, theta)
    return y, (x, theta)

def F_bwd(res, g):
    x, theta = res
    pf_ptheta = jax.grad(f, argnums=1)(x, theta)
    pf_px     = jax.grad(f, argnums=0)(x, theta)
    A = A_param(theta)

    psi = solve_transpose(A, pf_px)    
    _, vj_f = jax.vjp(partial(residual,x),theta)    

    vjp_=  vj_f( - psi)
    return (g * (pf_ptheta + vjp_[0]),)

def g(f, theta):
    return theta[0] ** f + theta[1]

F.defvjp(F_fwd, F_bwd)

print("JAX direct gradient")
G_theta_jax = lambda t : g(f(solve(A_param(t), b_param(t)), t), t)

print("\tG:        ",    G_theta_jax(theta_base))
print("\tdG_dtheta :",    jax.grad(G_theta_jax)(theta_base))

print("JAX custom VJP gradient")
G_theta_custom_vjp = lambda t : g(F(t), t)
print("\tG:        ",   G_theta_custom_vjp(theta_base))
print("\tdG_dtheta :",    jax.grad(G_theta_custom_vjp)(theta_base))


JAX direct gradient
	G:         0.8987822650513859
	dG_dtheta : [0.5098907  0.49111969]
JAX custom VJP gradient
	G:         0.8987822650513859
	dG_dtheta : [0.5098907  0.49111969]


Actually, JAX implements this machinery directly through jax.lax.custom_linear_solve and this can be used as well (and is in fact internally for jnp.linalg.solve!)

Here we make a little non differentiable solve that errors out whenever one tries to use AD on it.

Then we use the jax.lax.custom_linear_solve on it, this obtains the same result as our previous method. Note that it does *not* differentiate through the solver; that would have produced an error

In [142]:
@jax.custom_vjp
def _non_differentiable_solve(A, b, transpose = False):

        
    luff = jax.scipy.linalg.lu_factor(jax.lax.stop_gradient(A))
    
    return jax.scipy.linalg.lu_solve(luff, b)

def _non_differentiable_solve_fwd(A, b, transpose):
    x = _non_differentiable_solve(A, b, transpose)
    return x, (A, x, transpose)
def _non_differentiable_solve_bwd(res, g):  
    raise NotImplementedError("This solve is non-differentiable")

_non_differentiable_solve.defvjp(_non_differentiable_solve_fwd, _non_differentiable_solve_bwd)


def do_custom_linear_solve(theta):
    A = A_param(theta)
    b = b_param(theta)
    return jax.lax.custom_linear_solve(matvec = lambda x : A @ x, b=b, solve = lambda _, x : _non_differentiable_solve(A,x), transpose_solve = lambda _, x : _non_differentiable_solve(A.T, x))    

print("Custom non differentiable solve:")
try:
    g_non_diff = lambda t : g(f(_non_differentiable_solve(A_param(t), b_param(t)), t), t)
    print("\tG          \t:",    g_non_diff(theta_base))
    
    print("\tdG_dtheta  \t:",     jax.grad(g_non_diff)(theta_base))
except NotImplementedError as e:
    
    print("\tdG_dtheta  \t:","NotImplementedError")
    print("\t\tERROR:", e)

print("Custom linear solve")
g_diff = lambda t : g(f(do_custom_linear_solve(t), t), t)
print("\tG              \t:",  g_diff(theta_base))
print("\tdG_dtheta      \t: ", jax.grad(g_diff)(theta_base))

Custom non differentiable solve:
	G          	: 0.8987822650513859
	dG_dtheta  	: NotImplementedError
		ERROR: This solve is non-differentiable
Custom linear solve
	G              	: 0.8987822650513859
	dG_dtheta      	:  [0.5098907  0.49111969]


So why don't we just use that directly?

1. It does not allow Pytrees

In [156]:
from dataclasses import dataclass

@jax.tree_util.register_dataclass
@dataclass(frozen=True)
class A:
    A_mat : jnp.ndarray
    static_param : int

    def __call__(self, x):
        return self.A_mat @ x
    
    def solve(self, b):        
        
        return _non_differentiable_solve(self.A_mat, b)
    def solve_T(self, b):
        return _non_differentiable_solve(self.A_mat.T, b)

def do_custom_linear_solve_A(theta):
    A_mat = A_param(theta)
    A_obj = A(A_mat = A_mat, static_param = 42)
    b = b_param(theta)
    
    return jax.lax.custom_linear_solve(matvec = lambda x :  A_obj(x), b=b, solve = lambda _, x : A_obj.solve(x), transpose_solve = lambda _, x : A_obj.solve_T(x))

do_custom_linear_solve_A(theta_base)

Array([0.26167008, 0.47428538], dtype=float64)